In [7]:
from pathlib import Path
import os, stat, glob, traceback
import numpy as np
import pandas as pd
import arviz as az
from arviz_stats.base.array import array_stats
from cmdstanpy import CmdStanModel
import matplotlib.pyplot as plt

### Setup

In [8]:
# Paths and helper to compile models
stan_dir = Path('../stan').resolve()
out_root = Path.cwd() / 'stan_output'
out_root.mkdir(parents=True, exist_ok=True)
models = {
    'minimal': {'file': stan_dir / 'minimal.stan', 'outdir': out_root / 'minimal'},
    'province': {'file': stan_dir / 'logistic_province.stan', 'outdir': out_root / 'province'},
    'full': {'file': stan_dir / 'logistic_full.stan', 'outdir': out_root / 'full'},
    'response': {'file': stan_dir / 'logistic_response.stan', 'outdir': out_root / 'response'},
}
for m in models.values():
    m['outdir'].mkdir(parents=True, exist_ok=True)

def compile_model(stan_path):
    print('Compiling', stan_path.name)
    mdl = CmdStanModel(stan_file=str(stan_path))
    mode = os.stat(mdl.exe_file).st_mode
    os.chmod(mdl.exe_file, mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    return mdl

In [9]:
idata_min = az.from_netcdf(str(models['minimal']['outdir'] / 'idata.nc'))
idata_prov = az.from_netcdf(str(models['province']['outdir'] / 'idata.nc'))
idata_full = az.from_netcdf(str(models['full']['outdir'] / 'idata.nc'))
print('Loaded saved idata for: minimal, province, full')

Loaded saved idata for: minimal, province, full


In [10]:
model_idata = {
    "minimal": idata_min,
    "province": idata_prov,
    "full": idata_full,
}
print("Models:", list(model_idata.keys()))

Models: ['minimal', 'province', 'full']


In [11]:
# Quick inventory
rows = []
for name, idata in model_idata.items():
    has_posterior = hasattr(idata, "posterior")
    has_log_lik = hasattr(idata, "log_likelihood")
    has_ppc = hasattr(idata, "posterior_predictive")

    n_vars = len(idata.posterior.data_vars) if has_posterior else 0
    n_draws = (
        idata.posterior.sizes.get("chain", 1) * idata.posterior.sizes.get("draw", 0)
        if has_posterior else 0
    )

    rows.append({
        "model": name,
        "has_posterior": has_posterior,
        "has_log_likelihood": has_log_lik,
        "has_posterior_predictive": has_ppc,
        "n_posterior_vars": n_vars,
        "n_draws_total": int(n_draws),
    })

pd.DataFrame(rows).set_index("model")

,has_posterior,has_log_likelihood,has_posterior_predictive,n_posterior_vars,n_draws_total
model,,,,,
minimal,True,True,True,3,4000
province,True,True,True,4,4000
full,True,True,True,6,4000


In [12]:
loo_min = az.loo(model_idata['minimal'], pointwise=True)
print(loo_min)

Computed from 4000 posterior samples and 15203 observations log-likelihood matrix.

         Estimate       SE
elpd_loo -2246.76    73.78
p_loo        3.17        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)     15203  100.0%
   (0.70, 1]   (bad)          0    0.0%
    (1, Inf)   (very bad)     0    0.0%



In [13]:
loo_prov = az.loo(model_idata['province'], pointwise=True)
print(loo_prov)

Computed from 4000 posterior samples and 15203 observations log-likelihood matrix.

         Estimate       SE
elpd_loo -2018.97    66.52
p_loo       10.57        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)     15203  100.0%
   (0.70, 1]   (bad)          0    0.0%
    (1, Inf)   (very bad)     0    0.0%



In [14]:
loo_full = az.loo(model_idata['full'], pointwise=True)
print(loo_full)

Computed from 4000 posterior samples and 15203 observations log-likelihood matrix.

         Estimate       SE
elpd_loo -1957.04    67.69
p_loo       20.59        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)     15203  100.0%
   (0.70, 1]   (bad)          0    0.0%
    (1, Inf)   (very bad)     0    0.0%



In [15]:
comp = az.compare({
    'minimal': loo_min,
    'province': loo_prov,
    'full': loo_full
}, round_to='none')

comp

,rank,elpd,p,elpd_diff,weight,se,dse,warning
full,0,-1957.044269,20.592059,0.000000,9.470468e-01,67.689650,0.000000,False
province,1,-2018.968597,10.566749,61.924329,5.386240e-11,66.522598,18.057965,False
minimal,2,-2246.755781,3.166470,289.711513,5.295318e-02,73.775145,30.263059,False


In [16]:
az.summary(idata_full)

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
alpha,-2.61,0.53,-3.5,-1.8,1057,1576,1.00,0.016,0.011
a_prov[0],1.04,0.35,0.46,1.6,1069,1248,1.00,0.011,0.0077
a_prov[1],1.57,0.34,1,2.1,1062,1518,1.00,0.011,0.0075
a_prov[2],-0.3,0.36,-0.89,0.27,1159,1785,1.00,0.011,0.0075
a_prov[3],-0.86,0.75,-2.1,0.29,4098,2804,1.00,0.012,0.0085
a_prov[4],-0.98,0.75,-2.2,0.17,4199,2925,1.00,0.012,0.0083
a_prov[5],-0.39,0.89,-1.9,1,4874,2836,1.00,0.013,0.009
a_prov[6],-0.6,0.49,-1.4,0.18,1909,2423,1.00,0.011,0.0083
a_prov[7],0.83,0.35,0.26,1.4,1062,1370,1.00,0.011,0.0075
a_prov[8],-0.28,0.44,-0.99,0.4,1683,2283,1.00,0.011,0.008
